# InnerControl: inference example

Generate images with an InnerControl-trained SD1.5 ControlNet (depth, LineArt or HED).

**Pretrained checkpoints are not released yet.** Train a ControlNet with the scripts in `train/` (see the README) and point `controlnet_path` below to the saved weights, e.g. `work_dirs/MultiGen/depth_reward_aligned_ema`.

![InnerControl qualitative comparison](images/qualitative.jpg)

## 📚 Import libraries

In [ ]:
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel
from diffusers.schedulers import UniPCMultistepScheduler, DDIMScheduler
from PIL import Image
from transformers import DPTImageProcessor, DPTForDepthEstimation
import torchvision.transforms.functional as F
import os
import torch
import gc


In [ ]:
#@title 📚 Load ControlNet { display-mode: "form" }

# @markdown Model parameters
controlnet_path = "work_dirs/MultiGen/hed_reward_aligned_ema"  # @param {type:"string"}
torch_dtype = "torch.float16"  # @param ["torch.float16", "torch.bfloat16", "torch.float32"]
device = 'cuda'   # @param ["cuda", "cpu"]
scheduler = 'unipc'   # @param ["unipc", "ddim"]

torch_dtype = getattr(torch, torch_dtype.split(".")[-1])

torch.cuda.empty_cache()
gc.collect()

if not os.path.exists(controlnet_path):
    raise FileNotFoundError(
        f"{controlnet_path} not found. Pretrained InnerControl weights are not released yet: "
        "train a ControlNet with train/aligned_{depth,hed,linedrawing}.sh and set controlnet_path."
    )

controlnet = ControlNetModel.from_pretrained(controlnet_path, torch_dtype=torch_dtype)

try:
  del pipe
except:
  pass

# Load Stable Diffusion pipeline
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch_dtype,
    safety_checker=None  # Optional: disable safety checker for faster inference
)

if scheduler == 'ddim':
    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
else:
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)

pipe.enable_model_cpu_offload()

In [ ]:
#@title 📚 Run inference { display-mode: "form" }
# @markdown ### Inference Parameters

try:
  del images
except:
  pass

torch.cuda.empty_cache()
gc.collect()

Image_path = "/content/0_hed.png"                # @param {type:"string"}
prompt = "diver at the great white wall in fiji" # @param {type:"string"}

num_inference_steps = 20  # @param {type:"integer"}
guidance_scale = 7.5  # @param {type:"number"}
seed = 2  # @param {type:"integer"}


resolution = 512
image = Image.open(Image_path)


condition = image.convert('RGB').resize((resolution, resolution), Image.Resampling.BICUBIC)


generator = torch.Generator(device=device).manual_seed(seed)

images = pipe(
              prompt=prompt,
              image=condition,
              num_inference_steps=num_inference_steps,
              guidance_scale=guidance_scale,
              negative_prompt='worst quality, low quality',
              generator=generator).images


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))  # Adjust figure size if needed

plt.subplot(1, 2, 1)
plt.title("Control")
plt.imshow(condition)
plt.axis("off")

plt.subplot(1, 2, 2)
plt.title("Generation")
plt.imshow(images[0])
plt.axis("off")

plt.tight_layout()
plt.show()